In [48]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


1º Passo: Preparação e Merge dos Dados

In [49]:
print("=" * 60)
print("ETAPA 1 — CARREGAMENTO E MERGE DOS DADOS")
print("=" * 60)

ETAPA 1 — CARREGAMENTO E MERGE DOS DADOS


In [50]:
try:

 base_mensal = pd.read_csv('03.BaseDPEvolucaoMensalCisp.csv', sep=';', encoding='iso-8859-1')
 delegacias = pd.read_csv('08.DP.csv', sep=',', encoding='utf-8')
 print(delegacias)
 print(base_mensal.columns)

except Exception as e:
    print(f'Erro ao processar dados: {e}')


     codDP                           nome  \
0        1   001ª DP - Praça da República   
1        4   004ª DP - Praça da República   
2        5            005ª DP - Mem de Sá   
3        6          006ª DP - Cidade Nova   
4        7         007ª DP - Santa Teresa   
..     ...                            ...   
132    159  159ª DP - Cachoeira de Macacu   
133    165          165ª DP - Mangaratiba   
134    166       166ª DP - Angra dos Reis   
135    167               167ª DP - Paraty   
136    168            168ª DP - Rio Claro   

                                              endereco  
0    Av. Presidente Vargas, 1100 - Centro, Rio de J...  
1    Av. Presidente Vargas, 1100 - Centro, Rio de J...  
2    Avenida Gomes Freire, 320 - Centro, Rio de Jan...  
3    Rua Professor Clementino Fraga, 77 - Centro, R...  
4    Rua Francisco de Castro, 5 - Santa Teresa, Rio...  
..                                                 ...  
132  Avenida Lord Baden Powel, 93 - Centro, Cachoei...  
133

In [51]:
# Fazer o MERGE das bases
# A atividade diz: cisp na base mensal e codDP na base de delegacias

dados_merge = pd.merge(base_mensal,delegacias,left_on='cisp',right_on='codDP',how='left')

#Tratar valores nulos, selecionei apenas as colunas importantes.

dados_merge = dados_merge[['roubo_celular','roubo_transeunte','nome']].dropna()
dados_merge['roubo_celular'] = pd.to_numeric(dados_merge['roubo_celular'], errors='coerce')
dados_merge['roubo_transeunte'] = pd.to_numeric(dados_merge['roubo_transeunte'], errors='coerce')

dados_merge = dados_merge.dropna()

dados_merge.head()

,roubo_celular,roubo_transeunte,nome
0,32,26,001ª DP - Praça da República
1,14,25,004ª DP - Praça da República
2,34,26,005ª DP - Mem de Sá
3,20,14,006ª DP - Cidade Nova
4,1,4,007ª DP - Santa Teresa


In [52]:
celular_array = np.array(dados_merge['roubo_celular'])
transeunte_array = np.array(dados_merge['roubo_transeunte'])

2° Passo: DIAGNÓSTICO ESTATÍSTICO


In [53]:
print("\n" + "=" * 60)
print("ETAPA 2 — DIAGNÓSTICO ESTATÍSTICO")
print("=" * 60)


ETAPA 2 — DIAGNÓSTICO ESTATÍSTICO


In [54]:
# Vamos criar uma função para calcular todas as métricas.
# Tendência central

def diagnostico(dados_array):

    media = np.mean(dados_array)
    mediana = np.median(dados_array)
    variancia = np.var(celular_array)

    moda = pd.Series(celular_array).mode()[0]

    # Dispersão
    desvio_padrao = np.std(celular_array)
    cv = (desvio_padrao / media * 100) if media != 0 else np.nan

    # Posição
    minimo = np.min(celular_array)
    maximo = np.max(celular_array)
    amplitude = maximo - minimo

    q1 = np.quantile(celular_array, 0.25)
    q2 = mediana
    q3 = np.quantile(celular_array, 0.75)

    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    # Distribuição
    assimetria = pd.Series(celular_array).skew()
    curtose = pd.Series(celular_array).kurt()

    delta = media - mediana
    
    resultado = {
        'media': media,
        'mediana': mediana,
        'variancia': variancia,
        'moda': moda,
        'desvio_padrao': desvio_padrao,
        'cv_%': cv,
        'minimo': minimo,
        'maximo': maximo,
        'amplitude': amplitude,
        'q1': q1,
        'q2_mediana': q2,
        'q3': q3,
        'iqr': iqr,
        'limite_inferior': limite_inferior,
        'limite_superior': limite_superior,
        'assimetria': assimetria,
        'curtose': curtose,
        'delta': delta,
    }

    # ---- Imprimir relatório ----
    print(f"\n{'─'*50}")
    print(f"{'─'*50}")
    print(f"  TENDÊNCIA CENTRAL")
    print(f"    Média             : {media:>10.2f}")
    print(f"    Mediana           : {mediana:>10.2f}")
    print(f"\n  DISPERSÃO")
    print(f"    Variância         : {variancia:>10.2f}")
    print(f"    Moda              : {moda:>10.2f}")
    print(f"    Desvio Padrão     : {desvio_padrao:>10.2f}")
    print(f"    Coef. Variação    : {cv:>9.1f}%")
    print(f"\n  POSIÇÃO E LIMITES")
    print(f"    Mínimo            : {minimo:>10.2f}")
    print(f"    Máximo            : {maximo:>10.2f}")
    print(f"    Amplitude         : {amplitude:>10.2f}")
    print(f"    Q1 (25%)          : {q1:>10.2f}")
    print(f"    Q2 / Mediana (50%): {q2:>10.2f}")
    print(f"    Q3 (75%)          : {q3:>10.2f}")
    print(f"    IQR               : {iqr:>10.2f}")
    print(f"    Limite Inferior   : {limite_inferior:>10.2f}")
    print(f"    Limite Superior   : {limite_superior:>10.2f}")
    print(f"\n  FORMATO DA DISTRIBUIÇÃO")
    print(f"    Assimetria        : {assimetria:>10.4f}")
    print(f"    Curtose           : {curtose:>10.4f}")
    print(f"\n  ANÁLISE DE DESLOCAMENTO")
    print(f"    Delta (Média - Mediana): {delta:>7.2f}")

    return resultado

In [55]:
print("\nDIAGNÓSTICO - ROUBO DE CELULAR")
estat_celular = diagnostico(dados_merge['roubo_celular'])

print("\nDIAGNÓSTICO - ROUBO A TRANSEUNTE")
estat_transeunte = diagnostico(dados_merge['roubo_transeunte'])


DIAGNÓSTICO - ROUBO DE CELULAR

──────────────────────────────────────────────────
──────────────────────────────────────────────────
  TENDÊNCIA CENTRAL
    Média             :       7.98
    Mediana           :       2.00

  DISPERSÃO
    Variância         :     182.97
    Moda              :       0.00
    Desvio Padrão     :      13.53
    Coef. Variação    :     169.5%

  POSIÇÃO E LIMITES
    Mínimo            :       0.00
    Máximo            :     205.00
    Amplitude         :     205.00
    Q1 (25%)          :       0.00
    Q2 / Mediana (50%):       2.00
    Q3 (75%)          :      11.00
    IQR               :      11.00
    Limite Inferior   :     -16.50
    Limite Superior   :      27.50

  FORMATO DA DISTRIBUIÇÃO
    Assimetria        :     3.3690
    Curtose           :    18.7726

  ANÁLISE DE DESLOCAMENTO
    Delta (Média - Mediana):    5.98

DIAGNÓSTICO - ROUBO A TRANSEUNTE

──────────────────────────────────────────────────
───────────────────────────────────────

Explicação para o Negócio — Delta + Assimetria:
Resposta:
A análise mostra que a média de roubos (7,98) é significativamente maior que a mediana (2,00), gerando um Delta de 5,98. Esse deslocamento indica que a média está sendo puxada para cima por alguns valores muito altos.
O Delta de 5,98 mostra que a média de roubos é bem maior que a mediana, o que indica que alguns meses tiveram números muito altos de roubos. A assimetria positiva (3,37) confirma que a maioria dos meses tem poucos roubos, mas existem alguns meses com picos muito elevados.

Isso revela que os dados são influenciados por meses extremamente violentos, que aumentam a média geral. Para o negócio, isso significa que o risco de roubo não é constante e pode aumentar bastante em determinados períodos ou regiões, o que é importante considerar na definição do preço do seguro.


3º PASSO: IDENTIFICAÇÃO DE ANOMALIAS (OUTLIERS)

In [56]:
print("\n" + "=" * 60)
print("ETAPA 3 — IDENTIFICAÇÃO DE ANOMALIAS (OUTLIERS)")
print("=" * 60)


ETAPA 3 — IDENTIFICAÇÃO DE ANOMALIAS (OUTLIERS)


In [57]:
limite_superior = estat_celular['limite_superior']
outliers_celular = dados_merge[dados_merge['roubo_celular'] > limite_superior]
outliers_transeunte = dados_merge[dados_merge['roubo_transeunte'] > limite_superior]
